<a href="https://colab.research.google.com/github/faisalepty/Sign-Language-CNN/blob/main/train_validate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

In [2]:
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([

    transforms.ToTensor(),

])

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kapillondhe/american-sign-language")

print("Path to dataset files:", path)

100%|██████████| 4.64G/4.64G [00:56<00:00, 88.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kapillondhe/american-sign-language/versions/1


In [6]:
path = kagglehub.dataset_download("kapillondhe/american-sign-language")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/kapillondhe/american-sign-language/versions/1


In [8]:
dataset = datasets.ImageFolder(path + "/ASL_Dataset/Train", transform=val_transforms)

n_total = len(dataset)
n_train = int(0.8*n_total)
n_val = int(n_total - n_train)

train_dataset, val_dataset = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

In [9]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [10]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.conv3 = nn.Conv2d(16, 16, 5)

        self.fc1 = nn.Linear(in_features=33856, out_features=200)
        self.fc2 = nn.Linear(in_features=200, out_features=84)
        self.fc3 = nn.Linear(in_features=84, out_features=29)
        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = self.dropout2(x)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [13]:
device = "cpu"
model = LeNet().to(device=device)
# model = SmallCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [14]:
from tqdm import tqdm

In [15]:
def train(model, train_loader):
  model.train()
  running_loss = 0.0
  for images, labels in tqdm(train_loader, desc="Training", leave=True):
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
  train_loss = running_loss / len(train_loader)
  return train_loss

In [16]:
def validate(model, val_loader):
  model.eval()
  val_loss = 0.0
  correct = 0
  total = 0
  with torch.no_grad():
      for images, labels in tqdm(val_loader, desc="validating", leave=True):
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          loss = criterion(outputs, labels)
          val_loss += loss.item()
          preds = outputs.argmax(dim=1)
          correct += (preds == labels).sum().item()
          total += labels.size(0)
  val_loss /= len(val_loader)
  val_acc = correct / total
  return val_loss, val_acc

In [17]:
import torch
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor()
]
)

def predict(image_path, model, transform, device):
    # 1. Load and transform the image
    img = Image.open(image_path)
    img_tensor = transform(img).unsqueeze(0) # Add batch dimension: [1, 3, 128, 128]
    img_tensor = img_tensor.to(device)

    # 2. Set model to evaluation mode
    model.eval()

    with torch.no_grad():
        output = model(img_tensor)
        # Apply softmax to get probabilities
        probabilities = F.softmax(output, dim=1)
        # Get the predicted class index
        conf, predicted = torch.max(probabilities, 1)

    # 3. Map index to label (ImageFolder usually maps alphabetically)
    # class_names = train_dataset.classes -> ['hotdog', 'not-hotdog']
    labels = dataset2.classes

    print(f"Prediction: {labels[predicted.item()]} ({conf.item()*100:.2f}%)")

# # Usage:
# image_path = "/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/Y_test.jpg"
# predict(image_path, model, transform, device)

In [18]:
epochs = 10
best_val_acc = 0.0

for epoch in range(epochs):
    train_loss = train(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), "best_model.pth")



    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Training:   0%|          | 2/1036 [00:29<4:10:04, 14.51s/it]


KeyboardInterrupt: 

In [ ]:
train()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

